# **1. Perkenalan Dataset**

Dataset yang digunakan: **Pima Indians Diabetes Dataset**.

Dataset ini berisi data medis dari 768 pasien wanita keturunan Pima Indian, dengan 8 fitur (jumlah kehamilan, glukosa, tekanan darah, dll) dan 1 target biner (`Outcome`): apakah pasien menderita diabetes (1) atau tidak (0).

Sumber dataset: https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv

# **2. Import Library**

Pada tahap ini kita mengimpor pustaka yang dibutuhkan untuk EDA, preprocessing, dan modelling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None)

# **3. Memuat Dataset**

Dataset dimuat langsung dari file CSV yang sudah disimpan pada folder `diabetes_raw`.

In [2]:
DATA_PATH = 'diabetes_raw/diabetes.csv'
df = pd.read_csv(DATA_PATH)
print('Ukuran dataset:', df.shape)
df.head()

Ukuran dataset: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


# **4. Exploratory Data Analysis (EDA)**

Beberapa hal yang kita cek pada tahap EDA:
1. Distribusi target (`Outcome`) - apakah dataset seimbang?
2. Missing value tersembunyi (banyak kolom medis punya nilai 0 yang sebenarnya berarti data hilang, misalnya Glukosa/Tekanan Darah tidak mungkin 0)
3. Korelasi antar fitur
4. Distribusi tiap fitur dan potensi outlier

In [5]:
# 1. Distribusi target
print(df['Outcome'].value_counts(normalize=True))
sns.countplot(x='Outcome', data=df)
plt.title('Distribusi Outcome (0 = Tidak Diabetes, 1 = Diabetes)')
plt.show()

Outcome
0    0.651042
1    0.348958
Name: proportion, dtype: float64


In [6]:
# 2. Cek nilai 0 yang mencurigakan (0 tidak masuk akal secara medis untuk kolom-kolom ini)
cols_no_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_no_zero:
    n_zero = (df[col] == 0).sum()
    print(f'{col}: {n_zero} baris bernilai 0 ({n_zero/len(df)*100:.1f}%)')

Glucose: 5 baris bernilai 0 (0.7%)
BloodPressure: 35 baris bernilai 0 (4.6%)
SkinThickness: 227 baris bernilai 0 (29.6%)
Insulin: 374 baris bernilai 0 (48.7%)
BMI: 11 baris bernilai 0 (1.4%)


In [7]:
# 3. Korelasi antar fitur
plt.figure(figsize=(9,7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Korelasi Antar Fitur')
plt.show()

In [8]:
# 4. Distribusi tiap fitur numerik
df.hist(figsize=(12,10), bins=20)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Berdasarkan hasil EDA, tahapan preprocessing yang dilakukan:
1. Mengganti nilai 0 yang tidak masuk akal secara medis (Glucose, BloodPressure, SkinThickness, Insulin, BMI) dengan median kolom tersebut
2. Split data menjadi train dan test
3. Standarisasi fitur numerik dengan `StandardScaler`

Langkah-langkah ini akan dikonversi menjadi fungsi otomatis pada file `automate_<nama-siswa>.py`.

In [9]:
df_clean = df.copy()
cols_no_zero = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in cols_no_zero:
    median_val = df_clean.loc[df_clean[col] != 0, col].median()
    df_clean[col] = df_clean[col].replace(0, median_val)

df_clean.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In [10]:
X = df_clean.drop(columns=['Outcome'])
y = df_clean['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print('X_train:', X_train_scaled.shape, '| X_test:', X_test_scaled.shape)

X_train: (614, 8) | X_test: (154, 8)


In [11]:
import os
os.makedirs('diabetes_preprocessing', exist_ok=True)

train_out = X_train_scaled.copy()
train_out['Outcome'] = y_train.values
test_out = X_test_scaled.copy()
test_out['Outcome'] = y_test.values

train_out.to_csv('diabetes_preprocessing/train.csv', index=False)
test_out.to_csv('diabetes_preprocessing/test.csv', index=False)
print('Data hasil preprocessing tersimpan di folder diabetes_preprocessing/')

Data hasil preprocessing tersimpan di folder diabetes_preprocessing/
